# 02 CURE
The algorithm is in `../cure.py` because notebooks 06, 07 and 08 use it too.

In [1]:
import sys
import time
import numpy as np
sys.path.append("..")          # cf_common.py and cure.py are in the folder above

from cure import cure
from cf_common import load_data, evaluate_clustering, save_results, assign_to_nearest

data = load_data()
K_VALUES = [10, 20, 40, 80]

SAMPLE_SIZE = 10_000
N_REPRESENTATIVES = 8     # representative points per cluster
SHRINK = 0.8              # 0 = no shrinking, 1 = all representatives at the cluster mean
MIN_CLUSTER_SIZE = 10     # smaller clusters (in the sample) are treated as outliers

476,451 users, 120,650 items, 10,000 users are evaluated


## CURE on a sample

In [2]:
rng = np.random.default_rng(579)
sample = rng.choice(data.n_users, size=min(SAMPLE_SIZE, data.n_users), replace=False)

start = time.perf_counter()
representatives = cure(data.embedding[sample], K_VALUES, N_REPRESENTATIVES, SHRINK, MIN_CLUSTER_SIZE)
cure_seconds = time.perf_counter() - start
print(f"CURE on {len(sample):,} users: {cure_seconds:.1f}s (one run gives all k)")

CURE on 10,000 users: 33.0s (one run gives all k)


## Assign all users and evaluate every k
The labels are saved in `cache/cure_labels.npz` for `09_view_clusters`.

In [3]:
summaries, per_user, all_labels = [], [], {}
for k in K_VALUES:
    start = time.perf_counter()
    rep_points, rep_owner = representatives[k]
    labels = rep_owner[assign_to_nearest(data.embedding, rep_points)]
    fit_seconds = cure_seconds + time.perf_counter() - start
    all_labels[f"k{k}"] = labels

    summary, users = evaluate_clustering(data, "cure", k, labels, fit_seconds)
    summaries.append(summary)
    per_user.append(users)

np.savez("cache/cure_labels.npz", **all_labels)

cure             k=10   P@10 = 0.1257   clustering   33.2s   CF   24.8s


cure             k=20   P@10 = 0.1228   clustering   33.3s   CF   22.7s


cure             k=40   P@10 = 0.1202   clustering   33.5s   CF   22.1s


cure             k=80   P@10 = 0.1153   clustering   33.9s   CF   19.8s


## Results

In [4]:
results = save_results("cure", summaries, per_user)
results[["k", "clusters", "largest cluster %", "P@5", "P@10", "R@10", "P@20", "R@20",
         "clustering s", "CF s (all users, estimated)"]].round(4)

saved results\cure.csv and results\cure_per_user.csv


,k,clusters,largest cluster %,P@5,P@10,R@10,P@20,R@20,clustering s,"CF s (all users, estimated)"
0,10,9,34.9316,0.1794,0.1257,0.1257,0.0825,0.1650,33.1959,1179.3584
1,20,14,32.9929,0.1756,0.1228,0.1228,0.0809,0.1618,33.3108,1080.5255
2,40,24,31.0462,0.1719,0.1202,0.1202,0.0799,0.1598,33.5145,1054.7924
3,80,46,27.5588,0.1628,0.1153,0.1153,0.0768,0.1536,33.8570,941.3008
